# Notebook 2: Tokenization & Embeddings

**Goal:** See where the vectors that attention operates on actually come from. This notebook is lighter than Notebook 1 -- mostly run the cells and look at the output, we'll discuss as we go.

**Time box: ~15 minutes.**

**Note:** This notebook downloads a small pretrained model (`distilgpt2`) and its tokenizer from HuggingFace the first time it runs, so it needs internet access and may take a minute the very first time. Run this ahead of the session if possible so the download is already cached.

## Setup

In [ ]:
# If transformers isn't installed, uncomment the next line:
# !pip install transformers -q

import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

print("Loaded:", MODEL_NAME)


## 1. Tokenization: text becomes subword pieces

GPT-2's tokenizer uses byte-pair encoding (BPE): common words often stay whole, but rarer or longer words get split into subword chunks. Run this on a few sentences, including one with an unusual word, and look at the pieces.

In [ ]:
sentences = [
    "The cat sat on the mat.",
    "Transformers are transforming NLP.",
    "Unbelievably, the antidisestablishmentarianism example still works.",
]

for s in sentences:
    token_ids = tokenizer.encode(s)
    pieces = [tokenizer.decode([t]) for t in token_ids]
    print(f"'{s}'")
    print("  ->", pieces)
    print()


**Try it yourself:** change `sentences` above to include a word you're curious about (a made-up word, a technical term, a name) and see how it gets split.

## 2. From tokens to embeddings

Each token ID maps to a vector in the model's embedding table. Let's pull out a few and compare them with cosine similarity -- related words should end up "closer" in this space than unrelated ones.

In [ ]:
embedding_table = model.get_input_embeddings()  # nn.Embedding layer

def word_vector(word):
    # naive: take the first token if the word splits into multiple pieces
    token_id = tokenizer.encode(" " + word)[0]
    return embedding_table.weight[token_id].detach()

def cosine_sim(a, b):
    return torch.nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

pairs = [
    ("dog", "puppy"),
    ("dog", "airplane"),
    ("king", "queen"),
    ("king", "banana"),
    ("happy", "joyful"),
    ("happy", "sad"),
]

for w1, w2 in pairs:
    sim = cosine_sim(word_vector(w1), word_vector(w2))
    print(f"cosine_sim({w1!r}, {w2!r}) = {sim:.3f}")


## 3. Nearest neighbors in embedding space

Pick a word and find which other tokens in the vocabulary sit closest to it in embedding space.

In [ ]:
def nearest_neighbors(word, k=8):
    target = word_vector(word)
    all_embeddings = embedding_table.weight.detach()
    sims = torch.nn.functional.cosine_similarity(target.unsqueeze(0), all_embeddings)
    top_k = torch.topk(sims, k + 1)  # +1 because the word itself will likely be top match
    results = []
    for idx, score in zip(top_k.indices.tolist(), top_k.values.tolist()):
        tok = tokenizer.decode([idx])
        results.append((tok, round(score, 3)))
    return results

nearest_neighbors("dog")


**Try it yourself:** change the word in the cell above. Try a few different kinds of words (a common noun, a name, a number) and see how the neighbor lists differ.

### Discussion checkpoints

- Why subwords instead of whole words or individual characters? What's the trade-off (think: vocabulary size vs. sequence length)?
- The similarity structure we just saw (dog close to puppy, far from airplane) wasn't hand-coded anywhere -- where did it come from?
- Keep this in mind for Notebook 3: these are the *same kind* of vectors that get fed into the attention mechanism you tweaked in Notebook 1.
